# Lesson 01 - 介紹 AI 代理

歡迎來到 **AI 代理初學者** 課程的第一課！

一個 **AI 代理** 是一個使用大型語言模型（LLM）作為其推理引擎的程式，並能在現實世界中採取 *行動*——呼叫 API、查詢資料庫或執行程式碼——以代表使用者達成目標。

在這個筆記本中，你將建立你的第一個代理：一個建議假期目的地的 **旅遊代理**。在過程中你將學會如何：

1. 使用 **Microsoft Agent Framework** 連接至 Azure AI Foundry 代理服務。
2. 給代理一個 **工具** —— 它可以呼叫的純 Python 函式。
3. 執行代理並檢查它的回應。
4. 一邊串流代理的回應，一邊逐字元接收。


## 設定

在執行此筆記本之前，請確保您已：

1. **擁有一個已部署聊天模型的 Azure AI Foundry 專案**（例如 `gpt-4o-mini`）。
2. **使用 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名稱。

下面的儲存格會安裝您需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 創建你的第一個代理人

代理人需要兩樣東西：

- **指令**，告訴它*它是誰*以及*如何行為*（系統提示）。
- **工具** — 使用 `@tool` 裝飾的 Python 函數，代理人可以調用這些函數來檢索資訊或執行操作。

下面我們定義了一個簡單的工具，返回熱門度假目的地的清單。當使用者詢問旅遊推薦時，代理人會使用這個工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 串流回應

為了更互動的體驗，你可以**串流**代理的回應。代理不需等候完整回覆，而是隨著生成即時傳出文字片段。這在聊天介面中特別有用，因為你想即時顯示輸出。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在這課程中你學會了如何：

- **建立一個提供者**，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent Service。
- **用 `@tool` 裝飾器定義工具**，讓代理人可以呼叫你的 Python 函式。
- **執行代理人**，帶入使用者訊息並輸出其回應。
- **串流回應**，以實時輸出。

下一課我們將更深入探索代理架構，並學習如何賦予代理人更強大的工具及多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件乃使用人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。應以原始語言文件為權威來源。對於重要資訊，建議採用專業人工翻譯。因使用此翻譯而引致的任何誤解或錯譯，我們概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
